# GKT Model — Training & Performance Analysis

Graph-based Knowledge Tracing (GKT) models student knowledge as a graph of concepts updated via GNN + GRU with an Erase/Add gate. This notebook trains the model and produces publication-ready figures exportable to PDF.

**Sections**
1. Setup & Configuration
2. Data Exploration
3. Graph Structure Visualizations *(GKT-specific)*
4. Training Run with Metric Capture
5. Training Curves
6. Model Evaluation (ROC, PR, Calibration, Confusion Matrix)
7. Knowledge State Analysis *(GKT-specific)*
8. Erase/Add Gate Activity *(GKT-specific)*
9. Per-Concept Performance
10. Hidden State Dimensionality Reduction
11. Cross-Fold Performance
12. PDF Export

In [ ]:
import sys, os

# ── paths ──────────────────────────────────────────────────────────────────
NOTEBOOK_DIR = os.path.abspath('.')
PYKT_DIR     = os.path.join(NOTEBOOK_DIR, 'pykt-toolkit')
DATA_DIR     = os.path.join(NOTEBOOK_DIR, 'data')
FIGURES_DIR  = os.path.join(NOTEBOOK_DIR, 'figures')
os.makedirs(FIGURES_DIR, exist_ok=True)

sys.path.insert(0, PYKT_DIR)

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.nn.functional import binary_cross_entropy
from torch.autograd import Variable

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.backends.backend_pdf import PdfPages
import seaborn as sns

from sklearn.metrics import (
    roc_auc_score, roc_curve, accuracy_score,
    precision_recall_curve, average_precision_score,
    confusion_matrix
)
from sklearn.calibration import calibration_curve
from sklearn.decomposition import PCA

matplotlib.rcParams.update({
    'figure.dpi': 150,
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'legend.fontsize': 10,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

DEVICE  = 'cuda' if torch.cuda.is_available() else 'cpu'
ALL_FIGS = []
print(f'Device: {DEVICE}')
print('Setup OK')

In [ ]:
# ── Hyperparameters — match the values used in your original training run ───
# (decoded from the directory name:
#  my_dataset_gkt_qid_saved_model_42_0_0.01_0.5_64_transition_0_0_64)
FOLD        = 0
HIDDEN_DIM  = 64        # was 32 — corrected to match original training
EMB_SIZE    = 64        # was 32 — corrected to match original training
GRAPH_TYPE  = 'transition'
LR          = 0.01      # was 1e-3 — corrected to match original training
DROPOUT     = 0.0       # 0.5 during training; 0.0 here so model.eval() gives deterministic output
EPOCHS      = 200       # paper default; only used when no checkpoint is found
BATCH_SIZE  = 16
MAX_SEQ_LEN = 200
EARLY_STOP  = 10

# ── Checkpoint ───────────────────────────────────────────────────────────────
# Option A — point to an external .ckpt file (e.g. copied from another machine).
#   The pykt training script saves to:
#   examples/saved_model/my_dataset_gkt_qid_saved_model_42_0_0.01_0.5_64_transition_0_0_64/qid_model.ckpt
#   Set PRETRAINED_CKPT to that path if you have it:
PRETRAINED_CKPT = None  # e.g. r'pykt-toolkit\examples\saved_model\my_dataset_gkt_qid_saved_model_42_0_0.01_0.5_64_transition_0_0_64\qid_model.ckpt'

# Option B — auto-save/reload: the notebook saves here after training and
#            reloads it automatically on every subsequent run.
SAVED_MODELS_DIR = os.path.join(NOTEBOOK_DIR, 'saved_models')
os.makedirs(SAVED_MODELS_DIR, exist_ok=True)

_AUTO_CKPT    = os.path.join(SAVED_MODELS_DIR, f'gkt_fold{FOLD}_{GRAPH_TYPE}_hd{HIDDEN_DIM}.ckpt')
_HISTORY_JSON = os.path.join(SAVED_MODELS_DIR, f'gkt_fold{FOLD}_{GRAPH_TYPE}_hd{HIDDEN_DIM}_history.json')


## 1. Data Exploration

In [ ]:
train_path = os.path.join(DATA_DIR, f'train_fold_{FOLD}.csv')
valid_path = os.path.join(DATA_DIR, f'valid_fold_{FOLD}.csv')
test_path  = os.path.join(DATA_DIR, f'test_fold_{FOLD}.csv')

df_train = pd.read_csv(train_path)
df_valid = pd.read_csv(valid_path)
df_test  = pd.read_csv(test_path)
df_all   = pd.concat([df_train, df_valid, df_test], ignore_index=True)

# Remap skill ids to 0-based contiguous integers
unique_skills = sorted(df_all['skill'].unique())
skill2idx     = {s: i for i, s in enumerate(unique_skills)}
NUM_C = len(unique_skills)

for df in [df_train, df_valid, df_test, df_all]:
    df['skill_idx'] = df['skill'].map(skill2idx)

print(f'Train : {len(df_train):,} interactions, {df_train["uid"].nunique():,} students')
print(f'Valid : {len(df_valid):,} interactions, {df_valid["uid"].nunique():,} students')
print(f'Test  : {len(df_test):,}  interactions, {df_test["uid"].nunique():,} students')
print(f'Concepts (skills): {NUM_C}')
print(f'Global accuracy  : {df_all["response"].mean():.2%}')
df_all.head(3)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# 1a — Interactions per concept
ax = axes[0, 0]
counts = df_all.groupby('skill_idx').size().sort_values(ascending=False)
plot_n = min(40, NUM_C)
ax.bar(range(plot_n), counts.values[:plot_n], color='#2196F3', edgecolor='white')
ax.set_xlabel('Concept ID (sorted by frequency)')
ax.set_ylabel('Number of interactions')
ax.set_title(f'Interaction frequency per concept (top {plot_n}/{NUM_C})')

# 1b — Correct response rate per concept
ax = axes[0, 1]
acc_per_skill = df_all.groupby('skill_idx')['response'].mean().sort_values()
colors = plt.cm.RdYlGn(acc_per_skill.values)
ax.barh(range(len(acc_per_skill)), acc_per_skill.values, color=colors, edgecolor='none')
ax.axvline(df_all['response'].mean(), color='black', linestyle='--', linewidth=1.5,
           label=f'Global mean {df_all["response"].mean():.2%}')
ax.set_xlabel('Correct response rate')
ax.set_ylabel('Concept (sorted by difficulty)')
ax.set_title('Correct response rate per concept')
ax.legend()
ax.set_xlim(0, 1)

# 1c — Sequence length distribution per student
ax = axes[1, 0]
seq_lens = df_all.groupby('uid').size()
ax.hist(seq_lens, bins=40, color='#4CAF50', edgecolor='white', alpha=0.85)
ax.axvline(seq_lens.mean(), color='black', linestyle='--', linewidth=1.5,
           label=f'Mean {seq_lens.mean():.1f}')
ax.axvline(MAX_SEQ_LEN, color='red', linestyle=':', linewidth=1.5,
           label=f'Max seq len {MAX_SEQ_LEN}')
ax.set_xlabel('Number of interactions per student')
ax.set_ylabel('Number of students')
ax.set_title('Student sequence length distribution')
ax.legend()

# 1d — Response distribution split
ax = axes[1, 1]
split_counts = df_all['response'].value_counts()
bars = ax.bar(['Incorrect (0)', 'Correct (1)'], split_counts[[0, 1]].values,
              color=['#E91E63', '#4CAF50'], edgecolor='white')
for bar, val in zip(bars, split_counts[[0, 1]].values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
            f'{val:,}\n({val/len(df_all):.1%})', ha='center', va='bottom', fontsize=10)
ax.set_ylabel('Number of interactions')
ax.set_title('Response class distribution')

fig.suptitle('Dataset Overview', fontsize=15)
fig.tight_layout()
fig.savefig(f'{FIGURES_DIR}/01_data_exploration.pdf', bbox_inches='tight')
ALL_FIGS.append(fig)
plt.show()

## 2. Graph Structure Visualizations *(GKT-specific)*

GKT's core innovation is modelling student knowledge as a *graph of concepts*. The adjacency matrix defines how information flows between concepts during GNN aggregation.

In [ ]:
import importlib.util

def _load_pykt_module(name, relative_path):
    """Load a single pykt source file directly, bypassing pykt/__init__.py
    (which cascades into tqdm/wandb/etc. that are not needed here)."""
    filepath = os.path.join(PYKT_DIR, 'pykt', relative_path)
    spec = importlib.util.spec_from_file_location(name, filepath)
    mod  = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

_gkt_utils         = _load_pykt_module('gkt_utils', os.path.join('models', 'gkt_utils.py'))
build_dense_graph      = _gkt_utils.build_dense_graph
build_transition_graph = _gkt_utils.build_transition_graph

# Build both graphs for comparison
dense_graph = build_dense_graph(NUM_C).numpy()

# Build transition graph from training data
concept_seqs = (
    df_train.sort_values('timestamp')
            .groupby('uid')['skill_idx']
            .apply(lambda x: ','.join(x.astype(str).tolist()))
            .reset_index()
            .rename(columns={'skill_idx': 'concepts'})
)
transition_graph = build_transition_graph(concept_seqs, NUM_C).numpy()

print(f'Dense graph      — min: {dense_graph.min():.4f}, max: {dense_graph.max():.4f}')
print(f'Transition graph — min: {transition_graph.min():.4f}, max: {transition_graph.max():.4f}')
print(f'Transition sparsity: {(transition_graph == 0).mean():.1%} zeros')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

show_n = min(NUM_C, 50)  # show top-50 for readability

# Dense graph heatmap
ax = axes[0]
im = ax.imshow(dense_graph[:show_n, :show_n], cmap='Blues', aspect='auto')
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
ax.set_title(f'Dense Adjacency Matrix (uniform weights)\n[{show_n}×{show_n} shown]')
ax.set_xlabel('Target concept')
ax.set_ylabel('Source concept')

# Transition graph heatmap
ax = axes[1]
# Sort by row-sum to reveal structure
row_sums = transition_graph[:show_n, :show_n].sum(axis=1)
order = np.argsort(-row_sums)
mat = transition_graph[:show_n, :show_n][np.ix_(order, order)]
im2 = ax.imshow(mat, cmap='YlOrRd', aspect='auto',
                vmin=0, vmax=np.percentile(mat[mat > 0], 95) if (mat > 0).any() else 1)
plt.colorbar(im2, ax=ax, fraction=0.046, pad=0.04)
ax.set_title(f'Transition Adjacency Matrix (row-normalised)\n[{show_n}×{show_n}, sorted by out-degree]')
ax.set_xlabel('Target concept')
ax.set_ylabel('Source concept')

fig.suptitle('GKT Concept Graph — Adjacency Matrices', fontsize=14)
fig.tight_layout()
fig.savefig(f'{FIGURES_DIR}/02_adjacency_matrices.pdf', bbox_inches='tight')
ALL_FIGS.append(fig)
plt.show()

In [ ]:
# Concept transition network — show top-K strongest edges
try:
    import networkx as nx
    HAS_NX = True
except ImportError:
    HAS_NX = False
    print('networkx not installed — skipping network graph. Run: pip install networkx')

if HAS_NX:
    TOP_K_EDGES = 60  # strongest transitions to display
    TOP_NODES   = 30  # concepts with highest total degree

    G = nx.DiGraph()
    flat = [(i, j, transition_graph[i, j])
            for i in range(NUM_C) for j in range(NUM_C)
            if transition_graph[i, j] > 0]
    flat.sort(key=lambda x: -x[2])
    top_edges = flat[:TOP_K_EDGES]

    nodes_in_top = set()
    for i, j, w in top_edges:
        nodes_in_top.update([i, j])
    nodes_in_top = list(nodes_in_top)[:TOP_NODES]

    G.add_nodes_from(nodes_in_top)
    for i, j, w in top_edges:
        if i in nodes_in_top and j in nodes_in_top:
            G.add_edge(i, j, weight=w)

    fig, ax = plt.subplots(figsize=(14, 10))
    pos  = nx.spring_layout(G, k=2.5, seed=42)
    deg  = dict(G.degree())
    node_sizes  = [deg.get(n, 1) * 120 + 100 for n in G.nodes()]
    edge_weights = [G[u][v]['weight'] for u, v in G.edges()]
    max_w = max(edge_weights) if edge_weights else 1
    edge_widths  = [3 * w / max_w + 0.3 for w in edge_weights]

    nx.draw_networkx_nodes(G, pos, node_size=node_sizes,
                           node_color=node_sizes, cmap='viridis', ax=ax, alpha=0.9)
    nx.draw_networkx_labels(G, pos, ax=ax, font_size=8, font_color='white')
    nx.draw_networkx_edges(G, pos, width=edge_widths,
                           edge_color=edge_weights, edge_cmap=plt.cm.Oranges,
                           ax=ax, arrows=True, arrowsize=15,
                           connectionstyle='arc3,rad=0.1')

    sm = plt.cm.ScalarMappable(cmap=plt.cm.Oranges,
                               norm=plt.Normalize(vmin=min(edge_weights), vmax=max_w))
    sm.set_array([])
    plt.colorbar(sm, ax=ax, label='Transition weight (row-normalised)', fraction=0.03)

    ax.set_title(f'Concept Transition Network (top-{TOP_K_EDGES} edges, {len(G.nodes())} concepts)\n'
                 f'Node size = degree · Edge width/color = transition strength', fontsize=13)
    ax.axis('off')
    fig.tight_layout()
    fig.savefig(f'{FIGURES_DIR}/03_concept_network.pdf', bbox_inches='tight')
    ALL_FIGS.append(fig)
    plt.show()

## 3. Training Run with Metric Capture

In [ ]:
_gkt_mod = _load_pykt_module('gkt', os.path.join('models', 'gkt.py'))
GKT = _gkt_mod.GKT

def build_sequences(df, max_len=MAX_SEQ_LEN):
    """Build padded (skill, response) sequences from a DataFrame."""
    seqs_c, seqs_r = [], []
    for _, group in df.sort_values('timestamp').groupby('uid'):
        c = group['skill_idx'].values
        r = group['response'].values
        if len(c) < 2:
            continue
        c, r = c[:max_len], r[:max_len]
        pad = max_len - len(c)
        c = np.concatenate([c, np.full(pad, -1, dtype=np.int64)])
        r = np.concatenate([r, np.full(pad, -1, dtype=np.int64)])
        seqs_c.append(c)
        seqs_r.append(r)
    return np.array(seqs_c, dtype=np.int64), np.array(seqs_r, dtype=np.int64)

train_c, train_r = build_sequences(df_train)
valid_c, valid_r = build_sequences(df_valid)
test_c,  test_r  = build_sequences(df_test)

print(f'Train sequences : {len(train_c)}')
print(f'Valid sequences : {len(valid_c)}')
print(f'Test sequences  : {len(test_c)}')


In [ ]:
def predict_on_sequences(model, c_arr, r_arr, batch_size=64):
    """Run forward pass and collect predictions + targets."""
    model.eval()
    all_preds, all_targets = [], []
    n = len(c_arr)
    with torch.no_grad():
        for start in range(0, n, batch_size):
            c_batch = torch.tensor(c_arr[start:start+batch_size], dtype=torch.long).to(DEVICE)
            r_batch = torch.tensor(r_arr[start:start+batch_size], dtype=torch.long).to(DEVICE)
            preds = model(c_batch, r_batch)   # [B, seq_len-1]
            # Build mask: next step is valid when shifted concept != -1
            shft_c = c_batch[:, 1:]            # [B, seq_len-1]
            shft_r = r_batch[:, 1:]            # [B, seq_len-1]
            mask   = (shft_c != -1) & (shft_r != -1)
            all_preds.append(preds[mask].cpu().numpy())
            all_targets.append(shft_r[mask].float().cpu().numpy())
    return np.concatenate(all_preds), np.concatenate(all_targets)

def compute_auc_acc(preds, targets):
    auc = roc_auc_score(targets, preds)
    acc = accuracy_score(targets, (preds >= 0.5).astype(int))
    return auc, acc

# Instantiate model
graph = torch.tensor(transition_graph if GRAPH_TYPE == 'transition' else dense_graph,
                     dtype=torch.float32)
model = GKT(
    num_c=NUM_C, hidden_dim=HIDDEN_DIM, emb_size=EMB_SIZE,
    graph_type=GRAPH_TYPE, graph=graph,
    dropout=DROPOUT, emb_type='qid'
).to(DEVICE)

optimizer = torch.optim.Adam(model.parameters(), lr=LR)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'GKT trainable parameters: {n_params:,}')

In [ ]:
import json

# ── Resolve which checkpoint to use ─────────────────────────────────────────
ckpt_path = PRETRAINED_CKPT if PRETRAINED_CKPT else _AUTO_CKPT

if os.path.exists(ckpt_path):
    # ── Load pre-trained model ───────────────────────────────────────────────
    print(f'Checkpoint found: {ckpt_path}')
    ckpt = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)

    # The checkpoint may have been saved by this notebook (dict with metadata)
    # or by pykt's train_model.py (plain state_dict).
    if isinstance(ckpt, dict) and 'model_state_dict' in ckpt:
        model.load_state_dict(ckpt['model_state_dict'])
        best_epoch = ckpt.get('best_epoch', '?')
        best_auc   = ckpt.get('best_auc',   float('nan'))
    else:
        # pykt saves a plain state_dict
        model.load_state_dict(ckpt)
        best_epoch, best_auc = '?', float('nan')

    print(f'Weights loaded  — best_epoch={best_epoch}, best_val_auc={best_auc}')

    # Load training history if present
    if os.path.exists(_HISTORY_JSON):
        with open(_HISTORY_JSON) as f:
            history = json.load(f)
        print(f'Training history loaded ({len(history["epoch"])} epochs)')
    else:
        history = {'epoch': [], 'train_loss': [], 'valid_auc': [], 'valid_acc': []}
        print('No training history file found — training curves will be skipped.')

else:
    # ── Train from scratch, then save ───────────────────────────────────────
    print(f'No checkpoint found at: {ckpt_path}')
    print('Training from scratch...\n')

    history = {'epoch': [], 'train_loss': [], 'valid_auc': [], 'valid_acc': []}
    best_auc, best_epoch, no_improve = 0.0, 0, 0
    best_state = None

    n_train = len(train_c)

    for epoch in range(1, EPOCHS + 1):
        model.train()
        idx = np.random.permutation(n_train)
        epoch_losses = []

        for start in range(0, n_train, BATCH_SIZE):
            batch_idx = idx[start:start + BATCH_SIZE]
            c_batch = torch.tensor(train_c[batch_idx], dtype=torch.long).to(DEVICE)
            r_batch = torch.tensor(train_r[batch_idx], dtype=torch.long).to(DEVICE)

            optimizer.zero_grad()
            preds  = model(c_batch, r_batch)
            shft_c = c_batch[:, 1:]
            shft_r = r_batch[:, 1:]
            mask   = (shft_c != -1) & (shft_r != -1)
            y, t   = preds[mask], shft_r[mask].float()
            if y.numel() == 0:
                continue
            loss = binary_cross_entropy(y.double(), t.double())
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            epoch_losses.append(loss.item())

        v_preds, v_targets = predict_on_sequences(model, valid_c, valid_r)
        v_auc, v_acc = compute_auc_acc(v_preds, v_targets)

        mean_loss = np.mean(epoch_losses)
        history['epoch'].append(epoch)
        history['train_loss'].append(mean_loss)
        history['valid_auc'].append(v_auc)
        history['valid_acc'].append(v_acc)

        if v_auc > best_auc:
            best_auc, best_epoch = v_auc, epoch
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1

        print(f'Epoch {epoch:3d} | loss {mean_loss:.5f} | val_auc {v_auc:.4f} | val_acc {v_acc:.4f}'
              + (' ← best' if epoch == best_epoch else ''))

        if no_improve >= EARLY_STOP:
            print(f'Early stopping at epoch {epoch} (best epoch: {best_epoch})')
            break

    # Restore best weights
    if best_state:
        model.load_state_dict({k: v.to(DEVICE) for k, v in best_state.items()})

    # ── Save checkpoint + history ────────────────────────────────────────────
    torch.save({
        'model_state_dict': model.state_dict(),
        'best_epoch': best_epoch,
        'best_auc':   best_auc,
        'num_c':      NUM_C,
        'hidden_dim': HIDDEN_DIM,
        'emb_size':   EMB_SIZE,
        'graph_type': GRAPH_TYPE,
        'fold':       FOLD,
    }, ckpt_path)

    with open(_HISTORY_JSON, 'w') as f:
        json.dump(history, f, indent=2)

    print(f'\nModel saved  → {ckpt_path}')
    print(f'History saved→ {_HISTORY_JSON}')

print(f'\nBest validation AUC: {best_auc:.4f} at epoch {best_epoch}')


## 4. Training Curves

In [ ]:
if not history['epoch']:
    print('No training history available (model was loaded from a checkpoint without history).')
    print(f'Checkpoint: {ckpt_path}')
    print('Training curves will not be shown. Re-train by deleting the checkpoint and re-running.')
else:
    hist_df = pd.DataFrame(history)

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))

    # Loss
    ax = axes[0]
    ax.plot(hist_df['epoch'], hist_df['train_loss'], color='#E91E63', linewidth=2, label='Train loss')
    if isinstance(best_epoch, int):
        ax.axvline(best_epoch, color='gray', linestyle='--', linewidth=1, label=f'Best epoch {best_epoch}')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Binary Cross-Entropy')
    ax.set_title('Training Loss')
    ax.legend()

    # Validation AUC
    ax = axes[1]
    ax.plot(hist_df['epoch'], hist_df['valid_auc'], color='#2196F3', linewidth=2, label='Val AUC')
    if isinstance(best_epoch, int):
        ax.axvline(best_epoch, color='gray', linestyle='--', linewidth=1, label=f'Best epoch {best_epoch}')
        ax.scatter([best_epoch], [best_auc], color='gold', s=80, zorder=5, label=f'Best AUC {best_auc:.4f}')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('AUC')
    ax.set_title('Validation AUC')
    ax.legend()

    # Validation Accuracy
    ax = axes[2]
    ax.plot(hist_df['epoch'], hist_df['valid_acc'], color='#4CAF50', linewidth=2, label='Val Accuracy')
    if isinstance(best_epoch, int):
        ax.axvline(best_epoch, color='gray', linestyle='--', linewidth=1, label=f'Best epoch {best_epoch}')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Accuracy')
    ax.set_title('Validation Accuracy')
    ax.legend()

    fig.suptitle(f'GKT Training Curves — Fold {FOLD}', fontsize=14)
    fig.tight_layout()
    fig.savefig(f'{FIGURES_DIR}/04_training_curves.pdf', bbox_inches='tight')
    ALL_FIGS.append(fig)
    plt.show()


## 5. Model Evaluation

In [ ]:
test_preds, test_targets = predict_on_sequences(model, test_c, test_r)
test_auc = roc_auc_score(test_targets, test_preds)
test_acc = accuracy_score(test_targets, (test_preds >= 0.5).astype(int))
print(f'Test AUC      : {test_auc:.4f}')
print(f'Test Accuracy : {test_acc:.4f}')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# ROC curve
ax = axes[0, 0]
fpr, tpr, _ = roc_curve(test_targets, test_preds)
ax.plot(fpr, tpr, color='#2196F3', linewidth=2, label=f'GKT (AUC = {test_auc:.4f})')
ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random (AUC = 0.50)')
ax.fill_between(fpr, tpr, alpha=0.08, color='#2196F3')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve')
ax.legend()

# Precision-Recall curve
ax = axes[0, 1]
prec, rec, _ = precision_recall_curve(test_targets, test_preds)
ap = average_precision_score(test_targets, test_preds)
baseline = test_targets.mean()
ax.plot(rec, prec, color='#FF9800', linewidth=2, label=f'GKT (AP = {ap:.4f})')
ax.axhline(baseline, color='gray', linestyle='--', linewidth=1, label=f'Baseline ({baseline:.2f})')
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curve')
ax.legend()

# Calibration
ax = axes[1, 0]
frac_pos, mean_pred_val = calibration_curve(test_targets, test_preds, n_bins=15, strategy='quantile')
ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Perfect calibration')
ax.plot(mean_pred_val, frac_pos, 'o-', color='#9C27B0', linewidth=2, markersize=5, label='GKT')
ax.fill_between(mean_pred_val, frac_pos, mean_pred_val,
                where=(frac_pos > mean_pred_val), alpha=0.1, color='#4CAF50', label='Overconfident')
ax.fill_between(mean_pred_val, frac_pos, mean_pred_val,
                where=(frac_pos < mean_pred_val), alpha=0.1, color='#E91E63', label='Underconfident')
ax.set_xlabel('Mean predicted probability')
ax.set_ylabel('Fraction of positives')
ax.set_title('Calibration Plot (Reliability Diagram)')
ax.legend()
ax.set_xlim(0, 1); ax.set_ylim(0, 1)

# Confusion matrix
ax = axes[1, 1]
cm = confusion_matrix(test_targets, (test_preds >= 0.5).astype(int))
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='Blues', ax=ax,
            xticklabels=['Pred 0', 'Pred 1'], yticklabels=['True 0', 'True 1'],
            linewidths=0.5, annot_kws={'size': 13})
# Add raw counts
for i in range(2):
    for j in range(2):
        ax.text(j + 0.5, i + 0.7, f'n={cm[i,j]:,}', ha='center', va='center',
                fontsize=9, color='gray')
ax.set_title('Confusion Matrix (row-normalised)')

fig.suptitle(f'GKT Model Evaluation — Test Set (Fold {FOLD})', fontsize=14)
fig.tight_layout()
fig.savefig(f'{FIGURES_DIR}/05_model_evaluation.pdf', bbox_inches='tight')
ALL_FIGS.append(fig)
plt.show()

## 6. Knowledge State Analysis *(GKT-specific)*

GKT maintains a hidden state `h_t ∈ ℝ^{num_c × hidden_dim}` — one vector per concept per student. We can track how the model's belief about each concept evolves as the student answers questions.

In [ ]:
def get_knowledge_states(model, c_seq, r_seq):
    """Run one student sequence and capture h_t (concept states) at each step."""
    model.eval()
    c_t = torch.tensor(c_seq, dtype=torch.long).unsqueeze(0).to(DEVICE)  # [1, T]
    r_t = torch.tensor(r_seq, dtype=torch.long).unsqueeze(0).to(DEVICE)

    features  = c_t * 2 + r_t
    batch_size, seq_len = features.shape
    ht = Variable(torch.zeros((1, model.num_c, model.hidden_dim), device=DEVICE))

    state_history = []   # list of [num_c, hidden_dim] arrays
    pred_history  = []   # list of [num_c] arrays
    qt_history    = []   # concept index at each step

    with torch.no_grad():
        for i in range(seq_len - 1):
            xt = features[:, i]
            qt = c_t[:, i]
            qt_mask = (qt != -1)
            if not qt_mask.any():
                break
            tmp_ht = model._aggregate(xt, qt, ht, 1)
            h_next, _, _, _ = model._update(tmp_ht, ht, qt)
            ht[qt_mask] = h_next[qt_mask]
            yt = model._predict(h_next, qt)

            state_history.append(ht[0].cpu().numpy())       # [num_c, hidden_dim]
            pred_history.append(yt[0].cpu().numpy())         # [num_c]
            qt_history.append(qt[0].item())

    return np.array(state_history), np.array(pred_history), qt_history

# Pick a test student with a long enough sequence
test_student_ids = (
    df_test.sort_values('timestamp').groupby('uid').size()
           .sort_values(ascending=False).index.tolist()
)

selected_uid = test_student_ids[0]
student_df   = df_test[df_test['uid'] == selected_uid].sort_values('timestamp')
s_concepts   = student_df['skill_idx'].values[:MAX_SEQ_LEN]
s_responses  = student_df['response'].values[:MAX_SEQ_LEN]

state_hist, pred_hist, qt_hist = get_knowledge_states(model, s_concepts, s_responses)
T = len(qt_hist)
print(f'Student {selected_uid}: {T} timesteps, '
      f'{len(np.unique(qt_hist))} unique concepts visited')

In [ ]:
# Knowledge state heatmap: P(correct) per concept over time
# pred_hist[t, c] = predicted probability for concept c at time t

# Focus on concepts that appear in this student's sequence
active_concepts = sorted(set(qt_hist))
active_concepts = active_concepts[:min(30, len(active_concepts))]

pred_matrix = pred_hist[:, active_concepts].T  # [n_active, T]

fig, axes = plt.subplots(2, 1, figsize=(16, 10), gridspec_kw={'height_ratios': [4, 1]})

ax = axes[0]
im = ax.imshow(pred_matrix, aspect='auto', cmap='RdYlGn', vmin=0, vmax=1,
               interpolation='nearest')
plt.colorbar(im, ax=ax, label='P(correct) — concept knowledge probability')
ax.set_yticks(range(len(active_concepts)))
ax.set_yticklabels([f'C{c}' for c in active_concepts], fontsize=8)
ax.set_xlabel('Timestep')
ax.set_ylabel('Concept')
ax.set_title(f'Knowledge State Heatmap — Student {selected_uid}\n'
             f'(Concept × Time · green=mastered, red=not mastered)')

# Mark timesteps where this concept was tested
for t, c in enumerate(qt_hist):
    if c in active_concepts:
        row = active_concepts.index(c)
        correct = s_responses[t]
        marker_color = '#00FF00' if correct else '#FF0000'
        ax.scatter(t, row, marker='*' if correct else 'x',
                   color=marker_color, s=60, zorder=5, linewidths=1)

# Response timeline
ax2 = axes[1]
ax2.step(range(T), s_responses[:T], where='mid', color='#2196F3', linewidth=1.5)
ax2.fill_between(range(T), s_responses[:T], step='mid', alpha=0.3, color='#2196F3')
ax2.set_xlabel('Timestep')
ax2.set_ylabel('Response')
ax2.set_yticks([0, 1])
ax2.set_yticklabels(['Incorrect', 'Correct'])
ax2.set_title('Student response sequence (★ = correct tested, × = incorrect tested)')
ax2.set_xlim(0, T - 1)

fig.tight_layout()
fig.savefig(f'{FIGURES_DIR}/06_knowledge_state_heatmap.pdf', bbox_inches='tight')
ALL_FIGS.append(fig)
plt.show()

In [ ]:
# Knowledge trajectories: top-5 most visited concepts
from collections import Counter
top5 = [c for c, _ in Counter(qt_hist).most_common(5)]

fig, ax = plt.subplots(figsize=(14, 6))
colors5 = ['#2196F3', '#4CAF50', '#FF9800', '#E91E63', '#9C27B0']

for c, col in zip(top5, colors5):
    traj = pred_hist[:, c]
    ax.plot(range(T), traj, color=col, linewidth=1.8, label=f'Concept {c}', alpha=0.85)
    # Mark when this concept is tested
    tested_at = [t for t, q in enumerate(qt_hist) if q == c]
    for t in tested_at:
        mk = '^' if s_responses[t] == 1 else 'v'
        ax.scatter(t, pred_hist[t, c], marker=mk, color=col, s=70, zorder=5)

ax.axhline(0.5, color='gray', linestyle=':', linewidth=1, label='Decision threshold 0.5')
ax.set_xlabel('Timestep')
ax.set_ylabel('P(correct) — Predicted knowledge probability')
ax.set_title(f'GKT Knowledge Trajectories — Top-5 Most Visited Concepts\n'
             f'Student {selected_uid} · (▲ correct, ▼ incorrect answer)')
ax.legend()
ax.set_ylim(0, 1)

fig.tight_layout()
fig.savefig(f'{FIGURES_DIR}/07_knowledge_trajectories.pdf', bbox_inches='tight')
ALL_FIGS.append(fig)
plt.show()

## 7. Erase/Add Gate Activity *(GKT-specific)*

GKT uses an Erase & Add Gate (analogous to DKVMN) to update its memory. The erase gate decides *what to forget*, and the add gate decides *what new information to add*. We hook into the gate to visualise its activity per concept.

In [ ]:
erase_history = []  # list of [num_c, hidden_dim] arrays
add_history   = []  # list of [num_c, hidden_dim] arrays

def hook_erase_add(module, input, output):
    x = input[0]
    erase_gate = torch.sigmoid(module.erase(x))   # [mask_num, num_c, feature_dim]
    add_feat   = torch.tanh(module.add(x))         # [mask_num, num_c, feature_dim]
    erase_history.append(erase_gate.detach().cpu().numpy())
    add_history.append(add_feat.detach().cpu().numpy())

handle = model.erase_add_gate.register_forward_hook(hook_erase_add)

# Run student sequence through model
model.eval()
with torch.no_grad():
    c_t = torch.tensor(s_concepts, dtype=torch.long).unsqueeze(0).to(DEVICE)
    r_t = torch.tensor(s_responses, dtype=torch.long).unsqueeze(0).to(DEVICE)
    _ = model(c_t, r_t)

handle.remove()
print(f'Captured {len(erase_history)} gate activations')

In [ ]:
# Average erase and add gate magnitude per concept over time
# Each element in erase_history has shape [1, num_c, hidden_dim]

erase_means = np.array([e.mean(axis=0).mean(axis=-1) for e in erase_history])  # [T, num_c]
add_means   = np.array([a.mean(axis=0).mean(axis=-1) for a in add_history])    # [T, num_c]

# Collapse the mask_num dimension properly
erase_per_step = np.array([e.mean(axis=-1).mean(axis=0) for e in erase_history])  # [T, num_c]
add_per_step   = np.array([a.mean(axis=-1).mean(axis=0) for a in add_history])    # [T, num_c]

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Erase gate heatmap over time for active concepts
ax = axes[0]
eras_sub = erase_per_step[:, active_concepts[:20]].T  # [20, T]
im1 = ax.imshow(eras_sub, aspect='auto', cmap='Reds', vmin=0, vmax=1)
plt.colorbar(im1, ax=ax, fraction=0.046)
ax.set_xlabel('Timestep')
ax.set_ylabel('Concept')
ax.set_yticks(range(min(20, len(active_concepts))))
ax.set_yticklabels([f'C{c}' for c in active_concepts[:20]], fontsize=8)
ax.set_title('Erase Gate Activity\n(high=forgetting more)')

# Add gate heatmap over time
ax = axes[1]
add_sub = add_per_step[:, active_concepts[:20]].T
im2 = ax.imshow(add_sub, aspect='auto', cmap='Greens')
plt.colorbar(im2, ax=ax, fraction=0.046)
ax.set_xlabel('Timestep')
ax.set_ylabel('Concept')
ax.set_yticks(range(min(20, len(active_concepts))))
ax.set_yticklabels([f'C{c}' for c in active_concepts[:20]], fontsize=8)
ax.set_title('Add Gate Activity\n(high=adding more new info)')

# Average gate values per concept (bar chart)
ax = axes[2]
mean_erase = erase_per_step[:, active_concepts[:20]].mean(axis=0)
mean_add   = add_per_step[:, active_concepts[:20]].mean(axis=0)
x = np.arange(len(active_concepts[:20]))
width = 0.35
ax.bar(x - width/2, mean_erase, width, label='Erase gate (mean)', color='#E91E63', alpha=0.8)
ax.bar(x + width/2, mean_add,   width, label='Add gate (mean)',   color='#4CAF50', alpha=0.8)
ax.set_xlabel('Concept')
ax.set_ylabel('Mean gate activation')
ax.set_title('Mean Erase vs Add Gate per Concept')
ax.set_xticks(x)
ax.set_xticklabels([f'C{c}' for c in active_concepts[:20]], rotation=45, ha='right', fontsize=8)
ax.legend()

fig.suptitle(f'GKT Erase/Add Gate Analysis — Student {selected_uid}', fontsize=14)
fig.tight_layout()
fig.savefig(f'{FIGURES_DIR}/08_erase_add_gate.pdf', bbox_inches='tight')
ALL_FIGS.append(fig)
plt.show()

## 8. Per-Concept Performance

In [ ]:
def predict_with_concepts(model, c_arr, r_arr, batch_size=64):
    """Return (preds, targets, concept_ids) for all test interactions."""
    model.eval()
    all_preds, all_targets, all_concepts = [], [], []
    n = len(c_arr)
    with torch.no_grad():
        for start in range(0, n, batch_size):
            c_batch = torch.tensor(c_arr[start:start+batch_size], dtype=torch.long).to(DEVICE)
            r_batch = torch.tensor(r_arr[start:start+batch_size], dtype=torch.long).to(DEVICE)
            preds  = model(c_batch, r_batch)
            shft_c = c_batch[:, 1:]
            shft_r = r_batch[:, 1:]
            mask   = (shft_c != -1) & (shft_r != -1)
            all_preds.append(preds[mask].cpu().numpy())
            all_targets.append(shft_r[mask].float().cpu().numpy())
            all_concepts.append(shft_c[mask].cpu().numpy())
    return (np.concatenate(all_preds),
            np.concatenate(all_targets),
            np.concatenate(all_concepts))

t_preds, t_targets, t_concepts = predict_with_concepts(model, test_c, test_r)

per_concept_rows = []
for c in np.unique(t_concepts):
    mask = (t_concepts == c)
    y, p = t_targets[mask], t_preds[mask]
    if len(np.unique(y)) < 2 or len(y) < 5:
        continue
    per_concept_rows.append({
        'concept': int(c),
        'auc':     roc_auc_score(y, p),
        'acc':     accuracy_score(y, (p >= 0.5).astype(int)),
        'correct_rate': y.mean(),
        'n': int(mask.sum()),
    })

perf_df = pd.DataFrame(per_concept_rows).sort_values('auc', ascending=False)
print(f'Concepts with enough data: {len(perf_df)}')
perf_df.head(10)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# AUC per concept (horizontal bar, colour by value)
ax = axes[0]
show = perf_df.head(min(40, len(perf_df)))
colors = plt.cm.RdYlGn(show['auc'].values)
bars = ax.barh(range(len(show)), show['auc'].values, color=colors, edgecolor='white')
ax.axvline(0.5, color='red', linestyle='--', linewidth=1, label='Random')
ax.axvline(show['auc'].mean(), color='black', linestyle='--', linewidth=1.2,
           label=f'Mean {show["auc"].mean():.3f}')
ax.set_yticks(range(len(show)))
ax.set_yticklabels([f'C{c}' for c in show['concept']], fontsize=8)
ax.set_xlabel('AUC')
ax.set_title('AUC per Concept (sorted by AUC)')
ax.set_xlim(0, 1)
ax.legend()

# Scatter: concept difficulty vs AUC (sized by nb. observations)
ax = axes[1]
sc = ax.scatter(
    perf_df['correct_rate'], perf_df['auc'],
    s=perf_df['n'] / perf_df['n'].max() * 400 + 20,
    c=perf_df['auc'], cmap='RdYlGn', vmin=0.4, vmax=1.0,
    alpha=0.8, edgecolors='white', linewidth=0.5
)
plt.colorbar(sc, ax=ax, label='AUC')
ax.axhline(0.5, color='red', linestyle='--', linewidth=1)
ax.set_xlabel('Concept difficulty (correct response rate)')
ax.set_ylabel('AUC')
ax.set_title('Concept Difficulty vs. Prediction Quality\n(size = number of test observations)')

fig.suptitle('GKT Per-Concept Performance — Test Set', fontsize=14)
fig.tight_layout()
fig.savefig(f'{FIGURES_DIR}/09_per_concept_performance.pdf', bbox_inches='tight')
ALL_FIGS.append(fig)
plt.show()

## 9. Hidden State Dimensionality Reduction

GKT assigns each concept a `hidden_dim`-dimensional state vector. We use PCA to project these to 2D and visualise how concept representations cluster.

In [ ]:
# Collect all concept hidden states across multiple students
hidden_snapshots = []   # list of (concept_id, hidden_state, timestep, student)

sample_students = test_student_ids[:min(10, len(test_student_ids))]

for uid in sample_students:
    s_df = df_test[df_test['uid'] == uid].sort_values('timestamp')
    if len(s_df) < 5:
        continue
    sc = s_df['skill_idx'].values[:MAX_SEQ_LEN]
    sr = s_df['response'].values[:MAX_SEQ_LEN]

    s_states, _, s_qt = get_knowledge_states(model, sc, sr)

    # Sample every 5 steps to avoid redundancy
    for t in range(0, len(s_qt), 5):
        c_tested = s_qt[t]
        h = s_states[t, c_tested]   # [hidden_dim]
        hidden_snapshots.append({'concept': c_tested,
                                 'state': h,
                                 'timestep': t,
                                 'student': uid})

if hidden_snapshots:
    snap_states   = np.array([s['state'] for s in hidden_snapshots])
    snap_concepts = np.array([s['concept'] for s in hidden_snapshots])
    snap_times    = np.array([s['timestep'] for s in hidden_snapshots])

    pca = PCA(n_components=2, random_state=42)
    pca_coords = pca.fit_transform(snap_states)
    print(f'PCA explained variance: {pca.explained_variance_ratio_.sum():.1%}')
    print(f'Snapshots collected: {len(hidden_snapshots)}')

In [ ]:
if hidden_snapshots:
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))

    # Coloured by concept
    ax = axes[0]
    sc1 = ax.scatter(pca_coords[:, 0], pca_coords[:, 1],
                     c=snap_concepts, cmap='tab20',
                     s=25, alpha=0.6, edgecolors='none')
    plt.colorbar(sc1, ax=ax, label='Concept ID')
    ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
    ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
    ax.set_title('PCA of Concept Hidden States\n(coloured by concept ID)')

    # Coloured by timestep (early vs late)
    ax = axes[1]
    sc2 = ax.scatter(pca_coords[:, 0], pca_coords[:, 1],
                     c=snap_times, cmap='plasma',
                     s=25, alpha=0.6, edgecolors='none')
    plt.colorbar(sc2, ax=ax, label='Timestep')
    ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
    ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
    ax.set_title('PCA of Concept Hidden States\n(coloured by timestep — early=dark, late=bright)')

    fig.suptitle(f'GKT Hidden State Geometry — PCA over {len(sample_students)} students', fontsize=14)
    fig.tight_layout()
    fig.savefig(f'{FIGURES_DIR}/10_hidden_state_pca.pdf', bbox_inches='tight')
    ALL_FIGS.append(fig)
    plt.show()

## 10. Cross-Fold Performance

In [ ]:
# Quick cross-fold eval: train a short run on each fold and record test AUC/ACC
# Set QUICK_EPOCHS low for speed; increase for accurate comparison
QUICK_EPOCHS = 15

fold_results = []
for fold in range(5):
    tr_c, tr_r = build_sequences(pd.read_csv(os.path.join(DATA_DIR, f'train_fold_{fold}.csv')).assign(
        skill_idx=lambda d: d['skill'].map(skill2idx)))
    va_c, va_r = build_sequences(pd.read_csv(os.path.join(DATA_DIR, f'valid_fold_{fold}.csv')).assign(
        skill_idx=lambda d: d['skill'].map(skill2idx)))
    te_c, te_r = build_sequences(pd.read_csv(os.path.join(DATA_DIR, f'test_fold_{fold}.csv')).assign(
        skill_idx=lambda d: d['skill'].map(skill2idx)))

    m = GKT(num_c=NUM_C, hidden_dim=HIDDEN_DIM, emb_size=EMB_SIZE,
            graph_type=GRAPH_TYPE, graph=graph.clone(),
            dropout=DROPOUT, emb_type='qid').to(DEVICE)
    opt = torch.optim.Adam(m.parameters(), lr=LR)

    best_va, best_state_f = 0, None
    for ep in range(QUICK_EPOCHS):
        m.train()
        idx_f = np.random.permutation(len(tr_c))
        for st in range(0, len(tr_c), BATCH_SIZE):
            bi = idx_f[st:st+BATCH_SIZE]
            cb = torch.tensor(tr_c[bi], dtype=torch.long).to(DEVICE)
            rb = torch.tensor(tr_r[bi], dtype=torch.long).to(DEVICE)
            opt.zero_grad()
            pr = m(cb, rb)
            sc_b, sr_b = cb[:, 1:], rb[:, 1:]
            mk = (sc_b != -1) & (sr_b != -1)
            y, t = pr[mk], sr_b[mk].float()
            if y.numel() == 0: continue
            binary_cross_entropy(y.double(), t.double()).backward()
            torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0)
            opt.step()
        va_p, va_t = predict_on_sequences(m, va_c, va_r)
        va_auc = roc_auc_score(va_t, va_p)
        if va_auc > best_va:
            best_va = va_auc
            best_state_f = {k: v.cpu().clone() for k, v in m.state_dict().items()}

    if best_state_f:
        m.load_state_dict({k: v.to(DEVICE) for k, v in best_state_f.items()})
    te_p, te_t = predict_on_sequences(m, te_c, te_r)
    te_auc = roc_auc_score(te_t, te_p)
    te_acc = accuracy_score(te_t, (te_p >= 0.5).astype(int))
    fold_results.append({'fold': fold, 'test_auc': te_auc, 'test_acc': te_acc, 'val_auc': best_va})
    print(f'Fold {fold} — test_auc={te_auc:.4f} | test_acc={te_acc:.4f}')

fold_df = pd.DataFrame(fold_results)
print(f'\nMean AUC: {fold_df["test_auc"].mean():.4f} ± {fold_df["test_auc"].std():.4f}')
print(f'Mean ACC: {fold_df["test_acc"].mean():.4f} ± {fold_df["test_acc"].std():.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, metric, color, label in [
    (axes[0], 'test_auc', '#2196F3', 'Test AUC'),
    (axes[1], 'test_acc', '#4CAF50', 'Test Accuracy'),
]:
    vals  = fold_df[metric].values
    folds = fold_df['fold'].values
    bars  = ax.bar(folds, vals, color=color, edgecolor='white', alpha=0.85, width=0.6)
    mean, std = vals.mean(), vals.std()
    ax.axhline(mean, color='black', linestyle='--', linewidth=1.5,
               label=f'Mean {mean:.4f} ± {std:.4f}')
    ax.fill_between([-0.5, 4.5], mean - std, mean + std,
                    alpha=0.1, color='black')
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, val + 0.002,
                f'{val:.4f}', ha='center', va='bottom', fontsize=10)
    ax.set_xlabel('Fold')
    ax.set_ylabel(label)
    ax.set_title(f'{label} per Fold')
    ax.set_xticks(folds)
    ax.set_xticklabels([f'Fold {f}' for f in folds])
    ax.legend(fontsize=9)
    lo = max(0, mean - 4*std)
    hi = min(1, mean + 4*std)
    ax.set_ylim(lo - 0.02, hi + 0.05)

fig.suptitle(f'GKT Cross-Fold Performance (5-Fold, {QUICK_EPOCHS} epochs each)', fontsize=14)
fig.tight_layout()
fig.savefig(f'{FIGURES_DIR}/11_cross_fold_performance.pdf', bbox_inches='tight')
ALL_FIGS.append(fig)
plt.show()

## 11. GNN Aggregation Weight Distribution *(GKT-specific)*

At each step GKT computes weighted sums over concept neighbours. We visualise how the graph-weighted contributions are distributed across concepts.

In [ ]:
# Use the selected graph (transition or dense)
used_graph = transition_graph if GRAPH_TYPE == 'transition' else dense_graph

out_degrees  = used_graph.sum(axis=1)       # how strongly each concept influences others
in_degrees   = used_graph.sum(axis=0)       # how strongly each concept is influenced
weight_dist  = used_graph[used_graph > 0]   # non-zero edge weights

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

ax = axes[0]
ax.hist(weight_dist, bins=40, color='#2196F3', edgecolor='white', alpha=0.85)
ax.set_xlabel('Edge weight (transition probability)')
ax.set_ylabel('Count')
ax.set_title(f'Distribution of Non-zero Edge Weights\n({GRAPH_TYPE} graph)')
ax.axvline(weight_dist.mean(), color='black', linestyle='--', linewidth=1.5,
           label=f'Mean {weight_dist.mean():.4f}')
ax.legend()

ax = axes[1]
ax.bar(range(min(40, NUM_C)),
       np.sort(out_degrees)[::-1][:40], color='#FF9800', edgecolor='white')
ax.set_xlabel('Concept rank (by out-degree)')
ax.set_ylabel('Total out-weight')
ax.set_title('Top Concepts by GNN Out-Degree\n("teacher" concepts)')

ax = axes[2]
ax.bar(range(min(40, NUM_C)),
       np.sort(in_degrees)[::-1][:40], color='#9C27B0', edgecolor='white')
ax.set_xlabel('Concept rank (by in-degree)')
ax.set_ylabel('Total in-weight')
ax.set_title('Top Concepts by GNN In-Degree\n("student" concepts — most influenced)')

fig.suptitle(f'GKT Graph — GNN Aggregation Weight Analysis ({GRAPH_TYPE})', fontsize=14)
fig.tight_layout()
fig.savefig(f'{FIGURES_DIR}/12_gnn_weight_analysis.pdf', bbox_inches='tight')
ALL_FIGS.append(fig)
plt.show()

## 12. PDF Export

In [ ]:
output_pdf = os.path.join(FIGURES_DIR, 'gkt_analysis_report.pdf')

if ALL_FIGS:
    with PdfPages(output_pdf) as pdf:
        d = pdf.infodict()
        d['Title']   = 'GKT Model Analysis Report'
        d['Subject'] = 'Graph-based Knowledge Tracing'
        for fig in ALL_FIGS:
            pdf.savefig(fig, bbox_inches='tight')
    print(f'PDF exported: {output_pdf}')
    print(f'Total figures: {len(ALL_FIGS)}')
else:
    print('No figures collected. Re-run all cells from the top.')
    print('Alternative: jupyter nbconvert --to pdf gkt_analysis.ipynb')